In [ ]:
"""Convert PDF files to Markdown using Docling."""
 
from __future__ import annotations
 
import sys
from pathlib import Path
 
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import (
    PdfPipelineOptions,
    TableFormerMode,
    TableStructureOptions,
)
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling_core.types.doc.base import ImageRefMode
from docling_core.types.doc.document import ContentLayer

In [ ]:
def _pipeline_options() -> PdfPipelineOptions:
    # TableFormer ACCURATE gives best header/cell detection.
    # Add more enrichments here (formulas, code, etc.) as needed.
    return PdfPipelineOptions(
        do_table_structure=True,
        table_structure_options=TableStructureOptions(
            mode=TableFormerMode.ACCURATE,
            do_cell_matching=True,
        ),
        do_picture_classification=True,
        generate_picture_images=True,
    )
 
 
def _build_converter() -> DocumentConverter:
    return DocumentConverter(
        allowed_formats=[InputFormat.PDF],
        format_options={
            InputFormat.PDF: PdfFormatOption(pipeline_options=_pipeline_options())
        },
    )

In [ ]:
def _to_markdown(result, images_dir: Path) -> str:
    # Images are written to images_dir and referenced by relative path.
    # Switch to ImageRefMode.EMBEDDED for a single self-contained .md file.
    images_dir.mkdir(parents=True, exist_ok=True)
 
    md_out = images_dir.parent / f"{images_dir.stem.removesuffix('_images')}.md"
    result.document.save_as_markdown(
        md_out,
        included_content_layers={ContentLayer.BODY, ContentLayer.FURNITURE},
        page_break_placeholder="\n\n---\n\n",
        image_mode=ImageRefMode.REFERENCED,
        artifacts_dir=images_dir,
    )
    return md_out.read_text(encoding="utf-8")
 

In [ ]:

def convert(pdf_path: str | Path, output_dir: str | Path | None = None) -> Path:
    """Convert a PDF to Markdown and return the path of the .md file.
 
    Args:
        pdf_path: Path to the source PDF.
        output_dir: Destination folder. Defaults to the PDF's own directory.
 
    Returns:
        Path to the generated Markdown file.
    """
    pdf_path = Path(pdf_path).resolve()
    output_dir = Path(output_dir).resolve() if output_dir else pdf_path.parent
    output_dir.mkdir(parents=True, exist_ok=True)
 
    result = _build_converter().convert(str(pdf_path), raises_on_error=True)
 
    md_text = _to_markdown(result, images_dir=output_dir / f"{pdf_path.stem}_images")
    md_out = output_dir / f"{pdf_path.stem}.md"
    md_out.write_text(md_text, encoding="utf-8")
 
    return md_out

In [ ]:
def convert_batch(input_dir: str | Path, output_dir: str | Path) -> list[Path]:
    """Convert all PDFs in a directory. Skips files that fail.
 
    Args:
        input_dir: Folder containing .pdf files.
        output_dir: Destination folder for all outputs.
 
    Returns:
        List of successfully generated Markdown file paths.
    """
    pdfs = sorted(Path(input_dir).resolve().glob("*.pdf"))
    results = []
 
    for pdf in pdfs:
        try:
            results.append(convert(pdf, Path(output_dir) / pdf.stem))
        except Exception as exc:  # noqa: BLE001
            print(f"Skipped {pdf.name}: {exc}", file=sys.stderr)
 
    return results

In [ ]:
input_path = '/Users/nnandagopal/Desktop/personal_projects/RAG/data/SampleContract.pdf'
output_dir = 'output'

convert(input_path, output_dir)

In [ ]:
from markdowncleaner import MarkdownCleaner, CleanerOptions

def clean_rag_markdown(text: str) -> str:
    options = CleanerOptions()

    # --- Safe to enable for PDF-converted markdown ---
    options.fix_encoding_mojibake = True       # fixes garbled unicode
    options.normalize_quotation_symbols = True  # normalizes fancy quotes
    options.contract_empty_lines = True         # collapses 3+ blank lines → 1
    options.crimp_linebreaks = True             # fixes "infor-\nmation" → "information"
    options.remove_duplicate_headlines = True   # removes repeated headers from PDF pages

    # --- Disable anything that touches content length/sections ---
    options.remove_short_lines = False          # don't remove short lines — kills bullets/lists
    options.remove_sections = False             # keep ALL sections — you decide what's relevant
    options.remove_references_heuristically = False  # don't auto-remove references

    # --- Keep footnote removal on only if your docs don't use footnotes as content ---
    options.remove_footnotes_in_text = False    # safer off for non-academic docs

    cleaner = MarkdownCleaner(options=options)
    return cleaner.clean_markdown_string(text)


import re

def clean_repeated_special_chars(text: str) -> str:

    # --- PDF markdown escape noise ---
    text = re.sub(r'(\\_){2,}', '______', text)  # \_\_\_\_\_\_ → ______ (preserve blank line visually)
    text = re.sub(r'_{3,}',     '______', text)   # raw ___ → normalised blank

    text = re.sub(r'([ \t])\1{2,}', r'\1', text)  # repeated spaces/tabs → one
    text = re.sub(r'!{2,}',         '!',   text)
    text = re.sub(r'@{2,}',         '@',   text)
    text = re.sub(r'\${2,}',        '$',   text)
    text = re.sub(r'%{2,}',         '%',   text)
    text = re.sub(r'\^{2,}',        '^',   text)
    text = re.sub(r'&{2,}',         '&',   text)
    text = re.sub(r'={2,}',         '=',   text)
    text = re.sub(r'\+{2,}',        '+',   text)
    text = re.sub(r';{2,}',         ';',   text)
    text = re.sub(r':{2,}',         ':',   text)
    text = re.sub(r',{2,}',         ',',   text)
    text = re.sub(r'\?{2,}',        '?',   text)
    text = re.sub(r'/{2,}',         '/',   text)
    text = re.sub(r'[\\]{2,}',      '\\\\', text)  # fixed backslash pattern

    return text

# Usage
with open("output/SampleContract.md", "r") as f:
    raw = f.read()

markdown_text = clean_rag_markdown(raw)
markdown_text = clean_repeated_special_chars(markdown_text)


In [ ]:
markdown_text

In [ ]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

headers_to_split_on = [
    ("#",    "h1"),
    ("##",   "h2"),
    ("###",  "h3"),
    ("####", "h4"),  # include if your docs go this deep
]

md_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on,
    strip_headers=False  # keep the heading text inside the chunk content too
)

header_splits = md_splitter.split_text(markdown_text)

In [ ]:
for i in header_splits:
    print(i)
    print(i.metadata)
    print("---"*50)

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,        # tune to your embedding model's sweet spot
    chunk_overlap=150,      # overlap helps with boundary context
    keep_separator = False,
    separators=[
        "\n\n",   # paragraph break — try this first
        "\n",     # line break
        ". ",     # sentence
        ", ",     # clause
        " ",      # word (last resort before character)
        ""
    ]
)

final_chunks = []
for doc in header_splits:
    if len(doc.page_content) <= 1000:
        final_chunks.append(doc)          # section fits — keep as-is
    else:
        sub_chunks = text_splitter.split_documents([doc])
        final_chunks.extend(sub_chunks)   # metadata is preserved automatically

In [ ]:
for i in final_chunks:
    print(i)
    print("---"*40)